# FedTrace — 04: Grant Outlays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/04_grant_outlays.ipynb)

**Runtime:** CPU only  
**Input checkpoints:** `data/raw/02_grants.jsonl` — written by notebook 02  
**Publishes to GitHub:** `data/findings/04_grant_outlays.md`

**Driving questions:**
1. Which USASpending API endpoints can return outlay data for assistance awards?
2. If all endpoints fail, what explains the gap and what is the next viable path?

**Investigation log — all API paths exhausted:**
- `/api/v2/awards/{id}/` (notebook 02): `total_outlays` field present but null for all 12,361 assistance awards.
- `/api/v2/awards/funding/` (Section 2, Probe A): HTTP 200, zero result rows for all sampled grants. Endpoint does not index assistance awards.
- `/api/v2/search/spending_by_award/` with FAIN (Section 3, Probe B): zero results across all three assistance type groups (grants, direct_payments, loans). Consistent with cancelled awards having been purged from the USASpending search index — the direct award endpoint still resolves them, which confirms the records exist but are not search-indexed.

**Status:** confirmed methodology gap. Grant outlay data is not available via any USASpending API endpoint for these cancelled awards. The remaining path is the USASpending bulk download (`Assistance_PrimeTransactions` CSV files), which contains all records regardless of search-index status. See Section 4.

Current findings: [`data/findings/04_grant_outlays.md`](../data/findings/04_grant_outlays.md).

## Setup

Run cells 1–4 at the start of every session. They are idempotent — safe to re-run.

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('04_grant_outlays', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['raw_dir'].mkdir(parents=True, exist_ok=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
(PATHS['data_root'] / 'findings').mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import json
import re
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

USA_BASE = 'https://api.usaspending.gov/api/v2'
DELAY    = 0.25
BATCH    = 50

# Assistance award type codes grouped by API constraint.
# spending_by_award requires all codes in one request to belong to the same group.
ASSIST_TYPE_GROUPS = {
    'grants':          ['02', '03', '04', '05', '06', '09'],
    'direct_payments': ['10', '11'],
    'loans':           ['07', '08'],
}

# Fields to request from the search endpoint
SEARCH_FIELDS = [
    'Award ID', 'Recipient Name', 'Award Amount', 'Total Outlays',
    'Description', 'Awarding Agency',
    'Period of Performance Start Date',
    'Period of Performance Current End Date',
    'generated_internal_id',
]

_RETRYABLE = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {path}\n'
            'Run notebook 02 to completion before running this notebook.'
        )
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_jsonl_ids(path: Path, id_field: str) -> set:
    path = Path(path)
    if not path.exists():
        return set()
    ids = set()
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                ids.add(json.loads(line).get(id_field))
    return ids


def append_jsonl(path: Path, records: list[dict]) -> None:
    with open(path, 'a') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')


def extract_fain(generated_unique_award_id: str) -> str | None:
    """Extract FAIN from generated_unique_award_id.

    Format: ASST_{TYPE}_{FAIN}_{AGENCY_CODE}
    Example: ASST_NON_NA18OAR4320123_1330 -> NA18OAR4320123

    The agency code is the last underscore-delimited segment (4 alphanumeric chars).
    The type is the second segment (e.g. NON, AGG).
    Everything between is the FAIN — joined back with underscores if the FAIN itself
    contains underscores.
    """
    parts = generated_unique_award_id.split('_')
    if len(parts) < 4 or parts[0] != 'ASST':
        return None
    # parts[0]='ASST', parts[1]=TYPE, parts[2:-1]=FAIN segments, parts[-1]=AGENCY_CODE
    fain_parts = parts[2:-1]
    return '_'.join(fain_parts) if fain_parts else None

In [ ]:
# ── CELL 4: Paths ─────────────────────────────────────────────────────────────
grants_path        = PATHS['raw_dir'] / '02_grants.jsonl'
grant_outlays_path = PATHS['raw_dir'] / '04_grant_outlays.jsonl'
output_json_path   = PATHS['outputs_dir'] / '04_grant_outlays.json'
findings_md_path   = PATHS['data_root'] / 'findings' / '04_grant_outlays.md'

print('Input checkpoint state:')
n = sum(1 for l in grants_path.read_text().splitlines() if l.strip()) if grants_path.exists() else 0
status = 'OK' if grants_path.exists() else 'MISSING — run notebook 02'
print(f'  grants:         {n:,} records  [{status}]')

n_fetched = len(load_jsonl_ids(grant_outlays_path, 'award_id'))
print(f'  grant_outlays:  {n_fetched:,} records fetched so far')

## 1. Load Grant Award IDs

In [ ]:
grants_records = load_jsonl(grants_path)
grants_df      = pd.DataFrame(grants_records)

print(f'Grants loaded: {len(grants_df):,}')
print()

# Confirm the known gap
if 'total_outlays' in grants_df.columns:
    null_count = grants_df['total_outlays'].isna().sum()
    print(f'total_outlays from /awards/{{id}}/ endpoint: {null_count:,} null ({null_count/len(grants_df)*100:.1f}%)')

award_ids = grants_df['award_id'].dropna().tolist() if 'award_id' in grants_df.columns else []

# Extract FAINs and validate extraction coverage
grants_df['fain'] = grants_df['award_id'].apply(
    lambda x: extract_fain(str(x)) if pd.notna(x) else None
)
n_fain_ok = grants_df['fain'].notna().sum()
n_fain_fail = grants_df['fain'].isna().sum()
print(f'FAIN extraction: {n_fain_ok:,} ok, {n_fain_fail:,} failed')
if n_fain_fail > 0:
    print('Failed award_id samples:')
    print(grants_df[grants_df['fain'].isna()]['award_id'].head(5).tolist())

fains = grants_df['fain'].dropna().tolist()
# Build reverse map: FAIN -> award_id (for joining results back)
fain_to_award_id = dict(zip(grants_df['fain'].dropna(), grants_df.loc[grants_df['fain'].notna(), 'award_id']))

print(f'\nSample award_id -> FAIN:')
for aid, fain in list(zip(award_ids[:5], fains[:5])):
    print(f'  {aid} -> {fain}')

## 2. Probe A — Confirmed Dead End: `/awards/funding/`

**Result (run 2026-03-30):** `POST /api/v2/awards/funding/` returns HTTP 200 with zero result rows for all sampled grants. The endpoint does not index assistance awards. This path is closed.

The cell below is kept for reproducibility — re-running it will confirm the same result.

In [ ]:
PROBE_A_N = 5  # Reduced — already confirmed dead end; keeping for reproducibility
probe_a_ids = award_ids[:PROBE_A_N]

print(f'Probe A: POST /api/v2/awards/funding/ ({PROBE_A_N} samples)')
print('Expected: HTTP 200, 0 result rows (confirmed dead end for assistance awards)')
print()

probe_a_results = []
for award_id in probe_a_ids:
    try:
        r = requests.post(f'{USA_BASE}/awards/funding/', json={'award_id': award_id, 'limit': 10, 'page': 1}, timeout=15)
        data = r.json() if r.status_code == 200 else {}
        row_count = len(data.get('results', []))
        probe_a_results.append({'award_id': award_id, 'status': r.status_code, 'rows': row_count})
        print(f'  {award_id}: HTTP {r.status_code}, {row_count} rows')
    except Exception as e:
        probe_a_results.append({'award_id': award_id, 'status': None, 'rows': None})
        print(f'  {award_id}: Error — {e}')
    time.sleep(DELAY)

has_rows = any(r['rows'] and r['rows'] > 0 for r in probe_a_results)
print(f'\nResult: {"rows present — re-evaluate" if has_rows else "confirmed dead end (0 rows for all samples)"}')

## 3. Probe B — Confirmed Dead End: `spending_by_award` Search Endpoint

**Result (run 2026-03-30):** Zero results across all three assistance type groups. The search endpoint does not return these cancelled grants — consistent with cancelled awards being purged from USASpending's search index while remaining accessible via the direct award endpoint.

The cell below is kept for reproducibility.

In [ ]:
PROBE_B_N = 10
probe_b_fains = fains[:PROBE_B_N]

print(f'Probe B: spending_by_award with FAINs ({PROBE_B_N} samples)')
print(f'Award type groups: {list(ASSIST_TYPE_GROUPS.keys())}')
print(f'Sample FAINs: {probe_b_fains[:5]}')
print()

results = []

for group_name, type_codes in ASSIST_TYPE_GROUPS.items():
    payload = {
        'filters': {
            'award_ids':        probe_b_fains,
            'award_type_codes': type_codes,
        },
        'fields': SEARCH_FIELDS,
        'page': 1,
        'limit': PROBE_B_N * 2,
    }
    try:
        r = requests.post(f'{USA_BASE}/search/spending_by_award/', json=payload, timeout=30)
        print(f'{group_name} {type_codes}: HTTP {r.status_code}', end='')
        if r.status_code == 200:
            group_results = r.json().get('results', [])
            results.extend(group_results)
            print(f', {len(group_results)} results')
        else:
            print(f', response: {r.text[:200]}')
    except Exception as e:
        print(f'{group_name}: Error — {e}')
    time.sleep(DELAY)

print(f'\nTotal results across all groups: {len(results)}')

if results:
    print(f'\nFields in result[0]: {list(results[0].keys())}')

    outlays_vals = [r.get('Total Outlays') for r in results]
    n_non_null   = sum(1 for v in outlays_vals if v is not None)
    n_nonzero    = sum(1 for v in outlays_vals if v is not None and float(v) != 0)
    print(f'\nTotal Outlays: {n_non_null}/{len(results)} non-null, {n_nonzero}/{len(results)} non-zero')
    if n_non_null > 0:
        print(f'Sample values: {[v for v in outlays_vals if v is not None][:5]}')

    print(f'\nCross-check Award Amount vs total_obligation from notebook 02:')
    for result in results[:5]:
        returned_id   = result.get('Award ID', '')
        award_id_back = fain_to_award_id.get(returned_id)
        nb02_row      = grants_df[grants_df['award_id'] == award_id_back] if award_id_back else pd.DataFrame()
        nb02_obl      = float(nb02_row['total_obligation'].values[0]) if not nb02_row.empty else None
        returned_amt  = result.get('Award Amount')
        match_ok      = (
            nb02_obl is not None and returned_amt is not None
            and abs(float(returned_amt) - nb02_obl) / (abs(nb02_obl) or 1) < 0.01
        )
        print(
            f'  {returned_id}: '
            f'search={returned_amt}, nb02={nb02_obl}, '
            f'match={"ok" if match_ok else "MISMATCH or not found"}'
        )
else:
    print('No results from any group — FAINs may not be valid identifiers.')


## 4. Bulk Download Path

The `Assistance_PrimeTransactions` bulk download files contain all assistance award transactions regardless of search-index status. These files include transaction-level outlay data that can be aggregated to award level.

**Path:**
1. Use `POST /api/v2/bulk_download/awards/` with filters (award types, date range) to request a filtered download — avoids pulling the full dataset.
2. Alternatively, download the pre-built annual files from USASpending's S3 bucket.
3. Join on FAIN (`award_id_fain` field in the CSV) to match our 12,361 grants.
4. Aggregate `federal_action_obligation` and outlay fields per FAIN.

**Decision point:** The bulk download approach requires downloading multi-GB CSV files. Before building it, assess whether grant outlay data changes the accountability record materially — the ceiling and obligation record is complete. This section documents the path; implementation deferred pending that assessment.

**Run the cell below to publish the confirmed finding to GitHub.**

In [ ]:
# ── Publish confirmed gap finding ────────────────────────────────────────────
from scripts.colab_utils import publish_artifacts
from pathlib import Path

findings_md = """# Findings — 04: Grant Outlays

*Input: 12,361 matched grants (from notebook 02 checkpoint).*  
*Methodology: `notebooks/04_grant_outlays.ipynb`*

## Confirmed

**Grant outlay data is not available via any USASpending API endpoint for cancelled assistance awards.**

Three paths investigated and confirmed exhausted:

| Endpoint | Result |
| --- | --- |
| `GET /api/v2/awards/{id}/` | `total_outlays` field present but null for all 12,361 grants |
| `POST /api/v2/awards/funding/` | HTTP 200, zero result rows for all sampled grants |
| `POST /api/v2/search/spending_by_award/` (FAIN, all type groups) | Zero results across grants / direct_payments / loans groups |

**Working hypothesis:** Cancelled/terminated awards have been purged from USASpending's search index.
The direct award endpoint (`/awards/{id}/`) still resolves them — confirming the records exist — but
neither the search endpoint nor the funding endpoint indexes them. This would explain why
`total_obligation` is populated (direct DB lookup) while `total_outlays` is null and search returns nothing.

**Three-number record for grants — current state:**
- ceiling: present for all 12,361 matched grants (DOGE `value` field)
- obligated: present for all 12,361 matched grants (USASpending `total_obligation`)
- outlays: not available via any tested API path

**Aggregate totals (ceiling + obligated only):**
- ceiling:   $54.2B
- obligated: $35.7B
- outlays:   not available

## Open

- **Grant outlay data — remaining path:** USASpending bulk download (`Assistance_PrimeTransactions` CSV files).
  These files are not filtered by search-index status and contain all records including cancelled awards.
  Join path: FAIN (extracted from `generated_unique_award_id` as `ASST_{TYPE}_{FAIN}_{AGENCY_CODE}`) against `award_id_fain` field in the bulk CSV.
  Deferred pending assessment of whether grant outlays change the accountability record materially
  relative to the ceiling/obligation record already complete.
- **Linkage path for 22% of DOGE grants with no USASpending link** — unresolved separately.
"""

findings_md_path = PATHS['data_root'] / 'findings' / '04_grant_outlays.md'
findings_md_path.write_text(findings_md)
print(findings_md)

publish_artifacts(
    paths=['data/findings/04_grant_outlays.md'],
    message='Document grant outlay API gap — all endpoints exhausted',
    repo_dir=REPO,
)
